## Data Preparation for Geopolitical Sector Rotation Strategy
### Pulls data through 2026-06-26 from Yahoo Finance

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import os

warnings.filterwarnings('ignore')

from datetime import datetime, timedelta

# ============================================
# 1. Configuration
# ============================================

START_DATE = '2010-01-01'
END_DATE = datetime.today().strftime('%Y-%m-%d')  # Today's date

print("=" * 60)
print("GEOPOLITICAL SECTOR ROTATION - DATA PREPARATION")
print("=" * 60)
print(f"Data range: {START_DATE} to {END_DATE}")
print(f"Today's date: {datetime.today().strftime('%Y-%m-%d')}")
print()

# ============================================
# 1.5: Create data directory (FIX)
# ============================================

os.makedirs('data', exist_ok=True)
print("✓ data/ directory ready")
print()

# ============================================
# 2. Fetch VIX Data
# ============================================

print("1. Fetching VIX...")
vix_data = yf.download('^VIX', start=START_DATE, end=END_DATE, progress=False)

# Handle MultiIndex columns
if isinstance(vix_data.columns, pd.MultiIndex):
    if 'Close' in vix_data.columns.get_level_values(0):
        vix = vix_data['Close'].iloc[:, 0]
    else:
        vix = vix_data.iloc[:, 0]
elif 'Adj Close' in vix_data.columns:
    vix = vix_data['Adj Close']
elif 'Close' in vix_data.columns:
    vix = vix_data['Close']
else:
    vix = vix_data.iloc[:, 0]

if isinstance(vix, pd.DataFrame):
    vix = vix.iloc[:, 0]

vix = pd.Series(vix.values.flatten() if hasattr(vix.values, 'flatten') else vix.values, index=vix.index)
vix.name = 'VIX'
print(f"  ✓ VIX loaded: {len(vix)} days, range [{float(vix.min()):.1f}, {float(vix.max()):.1f}]")

# ============================================
# 3. Fetch Gold Data (GLD)
# ============================================

print("\n2. Fetching Gold (GLD)...")
gold_data = yf.download('GLD', start=START_DATE, end=END_DATE, progress=False)

if isinstance(gold_data.columns, pd.MultiIndex):
    if 'Close' in gold_data.columns.get_level_values(0):
        gold = gold_data['Close'].iloc[:, 0]
    else:
        gold = gold_data.iloc[:, 0]
elif 'Adj Close' in gold_data.columns:
    gold = gold_data['Adj Close']
elif 'Close' in gold_data.columns:
    gold = gold_data['Close']
else:
    gold = gold_data.iloc[:, 0]

if isinstance(gold, pd.DataFrame):
    gold = gold.iloc[:, 0]

gold = pd.Series(gold.values.flatten() if hasattr(gold.values, 'flatten') else gold.values, index=gold.index)
gold.name = 'Gold'
print(f"  ✓ Gold loaded: {len(gold)} days, range [{float(gold.min()):.2f}, {float(gold.max()):.2f}]")

# ============================================
# 4. Fetch All Sector ETFs
# ============================================

print("\n3. Fetching Sector ETFs...")
sector_etfs = {
    'XLK': 'Technology',
    'XLC': 'Communication Services',
    'XLY': 'Consumer Discretionary',
    'XLE': 'Energy',
    'XLI': 'Industrials',
    'XLV': 'Healthcare',
    'XLP': 'Consumer Staples',
    'XLB': 'Materials'
}

sector_prices = {}
for ticker, name in sector_etfs.items():
    print(f"  Fetching {ticker} ({name})...")
    data = yf.download(ticker, start=START_DATE, end=END_DATE, progress=False)
    
    if isinstance(data.columns, pd.MultiIndex):
        if 'Close' in data.columns.get_level_values(0):
            prices = data['Close'].iloc[:, 0]
        else:
            prices = data.iloc[:, 0]
    elif 'Adj Close' in data.columns:
        prices = data['Adj Close']
    elif 'Close' in data.columns:
        prices = data['Close']
    else:
        prices = data.iloc[:, 0]
    
    if isinstance(prices, pd.DataFrame):
        prices = prices.iloc[:, 0]
    
    sector_prices[ticker] = pd.Series(prices.values.flatten() if hasattr(prices.values, 'flatten') else prices.values, index=prices.index)

sector_prices = pd.DataFrame(sector_prices)
sector_returns = sector_prices.pct_change()
print(f"  ✓ All sectors loaded. Date range: {sector_prices.index[0].strftime('%Y-%m-%d')} to {sector_prices.index[-1].strftime('%Y-%m-%d')}")

# ============================================
# 5. Create Put/Call Proxy
# ============================================

print("\n4. Creating Put/Call proxy...")
put_call = 0.5 + (vix - vix.rolling(252).mean()) / (vix.rolling(252).std() * 3)
put_call = put_call.clip(0.3, 1.5)
print(f"  ✓ Put/Call proxy created: range [{put_call.min():.2f}, {put_call.max():.2f}]")

# ============================================
# 6. Data Freshness Check
# ============================================

print("\n" + "=" * 60)
print("DATA FRESHNESS CHECK")
print("=" * 60)

today = datetime.today()
data_sources = {
    'VIX': vix,
    'Gold (GLD)': gold,
    'XLK': sector_prices['XLK'],
    'XLC': sector_prices['XLC'],
    'XLY': sector_prices['XLY'],
    'XLE': sector_prices['XLE'],
    'XLI': sector_prices['XLI'],
    'XLV': sector_prices['XLV'],
    'XLP': sector_prices['XLP'],
    'XLB': sector_prices['XLB'],
}

for name, series in data_sources.items():
    last_date = series.dropna().index[-1]
    lag_days = (today - last_date).days
    status = "✓ CURRENT" if lag_days <= 2 else f"⚠ {lag_days} DAYS OLD"
    print(f"  {name:20s} last: {last_date.strftime('%Y-%m-%d')} ({status})")

print()

# ============================================
# 7. Save Prepared Data
# ============================================

print("5. Saving prepared data...")

# Save sector data
sector_prices.to_csv('data/sector_prices.csv')
sector_returns.to_csv('data/sector_returns.csv')

# Save VIX and Gold
vix.to_csv('data/vix.csv')
gold.to_csv('data/gold.csv')
put_call.to_csv('data/put_call.csv')

print("  ✓ Data saved to data/ directory")
print("    - sector_prices.csv")
print("    - sector_returns.csv")
print("    - vix.csv")
print("    - gold.csv")
print("    - put_call.csv")
print()

print("=" * 60)
print("DATA PREPARATION COMPLETE")
print("=" * 60)

GEOPOLITICAL SECTOR ROTATION - DATA PREPARATION
Data range: 2010-01-01 to 2026-06-27
Today's date: 2026-06-27

✓ data/ directory ready

1. Fetching VIX...
  ✓ VIX loaded: 4146 days, range [9.1, 82.7]

2. Fetching Gold (GLD)...
  ✓ Gold loaded: 4145 days, range [100.50, 495.90]

3. Fetching Sector ETFs...
  Fetching XLK (Technology)...
  Fetching XLC (Communication Services)...
  Fetching XLY (Consumer Discretionary)...
  Fetching XLE (Energy)...
  Fetching XLI (Industrials)...
  Fetching XLV (Healthcare)...
  Fetching XLP (Consumer Staples)...
  Fetching XLB (Materials)...
  ✓ All sectors loaded. Date range: 2010-01-04 to 2026-06-26

4. Creating Put/Call proxy...
  ✓ Put/Call proxy created: range [0.30, 1.50]

DATA FRESHNESS CHECK
  VIX                  last: 2026-06-26 (✓ CURRENT)
  Gold (GLD)           last: 2026-06-26 (✓ CURRENT)
  XLK                  last: 2026-06-26 (✓ CURRENT)
  XLC                  last: 2026-06-26 (✓ CURRENT)
  XLY                  last: 2026-06-26 (✓ CURRENT)